# Dummy Variables — Lecture Notebook
### Applied Statistical Data Analysis — Prof. Dr. Kristyna Ters | MSc Finance | FHNW
**Based on:** Brooks, C. — *Introductory Econometrics for Finance*, Cambridge University Press, Ch. 4

---
**Learning Objectives:**
- Encode qualitative information — events, regimes, categories, calendar effects — as **0/1 dummy variables**
- Interpret **intercept dummies** (level shifts) and **slope dummies / interactions** (sensitivity shifts)
- Avoid the **dummy-variable trap**: m categories → m − 1 dummies
- Run a **seasonality study** (day-of-week effect) as a joint F-test
- Test **structural breaks** with the dummy-based **Chow test** — by hand and with `f_test`

> Run each cell with **Shift+Enter**. This notebook accompanies the V7 lecture slides.

## Step 0 — Install & Import Libraries

In [ ]:
!pip install yfinance statsmodels --quiet

import yfinance as yf
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':'white', 'axes.facecolor':'white',
    'axes.spines.top':False, 'axes.spines.right':False,
    'axes.grid':True, 'grid.alpha':0.3, 'font.size':11
})
YELLOW = '#FDE70E'; ORANGE = '#FCB310'; RED = '#C70101'
GREY   = '#4B4B4B'; BLUE = '#0E75FE'; GREEN = '#0B7A3C'
print('✓ Libraries loaded.')

---
# Part 1 — Building Your First Dummy

A dummy (indicator) variable takes only the values 0 and 1:

$$D_t = 1 \text{ if the condition holds}, \qquad D_t = 0 \text{ otherwise.}$$

**You** define the condition — an event, a regime, a category, a calendar rule. Our running example: the COVID regime dummy, switching on 16 March 2020 (the week global markets hit their pandemic low).

### 1.1 Download data and build the dummy

In [ ]:
START, END = '2018-01-01', '2024-12-31'

px  = yf.download(['AAPL', '^GSPC'], start=START, end=END,
                  auto_adjust=True, progress=False)['Close']
ret = px.pct_change().dropna()
# rename by label, never by position: yfinance orders the Close columns alphabetically
ret = ret.rename(columns={'^GSPC': 'SP500'})[['AAPL', 'SP500']]

# The dummy: one comparison, converted to 0/1
ret['D'] = (ret.index >= '2020-03-16').astype(float)

print(f'Sample: {len(ret)} trading days, {ret.index[0].date()} → {ret.index[-1].date()}')
print(f'Days with D = 0 (pre-COVID):  {(ret["D"] == 0).sum()}')
print(f'Days with D = 1 (post-COVID): {(ret["D"] == 1).sum()}')
ret.head(3).round(4)

**Key point:** the dummy is *just another column* in the DataFrame — and it will be just another column in the X matrix. Estimation and inference work exactly as in the previous chapters; only the interpretation of its coefficient is new.

---
# Part 2 — Intercept Dummies: Shifting the Level

$$y_t = \beta_0 + \beta_1 x_t + \beta_2 D_t + u_t$$

- $D = 0$: $E[y] = \beta_0 + \beta_1 x$
- $D = 1$: $E[y] = (\beta_0 + \beta_2) + \beta_1 x$ — same slope, level shifted by $\beta_2$

Two **parallel** lines. $\beta_2$ is the ceteris-paribus level difference between the regimes, and testing $\beta_2 = 0$ is the usual t-test.

### 2.1 Estimate the intercept-dummy model

In [ ]:
X_lvl = sm.add_constant(ret[['SP500', 'D']])
m_lvl = sm.OLS(ret['AAPL'], X_lvl).fit(cov_type='HC1')

b2 = m_lvl.params['D']
print(f'β2_hat (level shift) = {b2:.6f}   t = {m_lvl.tvalues["D"]:.2f}   p = {m_lvl.pvalues["D"]:.3f}')
if abs(m_lvl.tvalues['D']) < 1.96:
    print('→ No significant LEVEL shift in Apple’s average excess performance after COVID.')
print('\n(The interesting change is in the SLOPE — that is Part 3.)')

---
# Part 3 — Slope Dummies: The Full Break Regression

Add the **interaction** $D_t \cdot x_t$:

$$r_{AAPL,t} = \beta_0 + \beta_1 r_{Mkt,t} + \beta_2 D_t + \beta_3 (D_t \cdot r_{Mkt,t}) + u_t$$

- $D = 0$: slope $= \beta_1$ (the **pre-COVID beta**)
- $D = 1$: slope $= \beta_1 + \beta_3$ (the **post-COVID beta**)

$\beta_3$ is the difference in slopes — did Apple's systematic risk change?

### 3.1 Estimate the break regression

In [ ]:
ret['D_x_SP500'] = ret['D'] * ret['SP500']

X_brk = sm.add_constant(ret[['SP500', 'D', 'D_x_SP500']])
m_brk = sm.OLS(ret['AAPL'], X_brk).fit(cov_type='HC1')
print(m_brk.summary())

In [ ]:
b1 = m_brk.params['SP500']
b3 = m_brk.params['D_x_SP500']

print(f'Pre-COVID beta:   β1_hat            = {b1:.4f}')
print(f'Slope shift:      β3_hat            = {b3:.4f}   (t = {m_brk.tvalues["D_x_SP500"]:.2f}, '
      f'p = {m_brk.pvalues["D_x_SP500"]:.4f})')
print(f'Post-COVID beta:  β1_hat + β3_hat   = {b1 + b3:.4f}')
print(f'\nRelative change: β3_hat / β1_hat = {b3/b1*100:+.1f}%')
print('(Note: the RELATIVE change divides the slope shift by the PRE-period beta.)')

### 3.2 See the break — one scatter, two regime lines

In [ ]:
pre  = ret[ret['D'] == 0]
post = ret[ret['D'] == 1]

fig, ax = plt.subplots(figsize=(9.5, 5.5))
ax.scatter(pre['SP500']*100,  pre['AAPL']*100,  s=8, color=GREY, alpha=0.4, label='pre-COVID (D = 0)')
ax.scatter(post['SP500']*100, post['AAPL']*100, s=8, color=BLUE, alpha=0.4, label='post-COVID (D = 1)')
xx = np.linspace(ret['SP500'].min(), ret['SP500'].max(), 50)
b0, b2 = m_brk.params['const'], m_brk.params['D']
ax.plot(xx*100, (b0 + b1*xx)*100, color='black', lw=2.2, label=f'pre:  slope = {b1:.2f}')
ax.plot(xx*100, (b0 + b2 + (b1 + b3)*xx)*100, color=RED, lw=2.2, label=f'post: slope = {b1 + b3:.2f}')
ax.axhline(0, color=GREY, lw=0.5); ax.axvline(0, color=GREY, lw=0.5)
ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1.02), ncol=4, frameon=False, fontsize=10)
ax.set_xlabel('S&P 500 daily return (%)'); ax.set_ylabel('AAPL daily return (%)')
ax.set_title('Apple’s market beta before and after COVID', fontweight='bold', loc='left', pad=30)
plt.tight_layout(); plt.show()

### 3.3 Test the break — two routes, same machinery

- **Single restriction** (t-test): $H_0: \beta_3 = 0$ — did the *slope* change?
- **Joint restrictions** (F-test): $H_0: \beta_2 = \beta_3 = 0$, $q = 2$ — was there *any* break at all?

In [ ]:
print('Route 1 — t-test on the interaction (H0: β3 = 0):')
print(f'  t = {m_brk.tvalues["D_x_SP500"]:.2f},  p = {m_brk.pvalues["D_x_SP500"]:.4f}')

print('\nRoute 2 — joint F-test (H0: β2 = β3 = 0):')
print(m_brk.f_test('D = D_x_SP500 = 0'))

---
# Part 4 — The Dummy-Variable Trap

Include a dummy for **every** category alongside the intercept, and the dummy columns sum to one in every row — exactly the intercept column. **Perfect multicollinearity**: $X^\top X$ cannot be inverted.

### 4.1 Demonstrate the trap numerically

In [ ]:
# Build ALL five weekday dummies for the S&P 500 sample (the wrong way!)
sp = ret[['SP500']].copy()
sp['weekday'] = sp.index.dayofweek        # 0 = Mon … 4 = Fri
all_five = pd.get_dummies(sp['weekday'], prefix='D').astype(float)
all_five.columns = ['D_Mon', 'D_Tue', 'D_Wed', 'D_Thu', 'D_Fri']

X_trap = sm.add_constant(all_five)        # constant + ALL five dummies
print('Row sums of the five dummy columns (first 5 rows):')
print(all_five.head().sum(axis=1).values, '  ← always exactly 1 = the intercept column')

rank = np.linalg.matrix_rank(X_trap.values)
print(f'\nColumns in X: {X_trap.shape[1]},  rank of X: {rank}')
print('→ rank < number of columns: X′X is singular — the model is not estimable.')
print('  (statsmodels silently switches to a pseudo-inverse; coefficients become arbitrary.)')

### 4.2 The fix — a reference category

Drop one category (`drop_first=True`): it becomes the **base**, and each dummy coefficient measures the *difference to that base*.

**Rule: with an intercept, use m − 1 dummies for m categories.** 5 weekdays → 4 dummies • 12 months → 11 dummies • 2 regimes → 1 dummy (the COVID dummy!).

In [ ]:
four = pd.get_dummies(sp['weekday'], prefix='D', drop_first=True).astype(float)
four.columns = ['D_Tue', 'D_Wed', 'D_Thu', 'D_Fri']   # Monday dropped → reference

X_ok = sm.add_constant(four)
print(f'Columns in X: {X_ok.shape[1]},  rank: {np.linalg.matrix_rank(X_ok.values)}   ← full rank ✓')
print('Monday is the reference: the constant is the Monday mean, each δ the difference to Monday.')

---
# Part 5 — Seasonality: The Day-of-Week Study

$$r_t = \alpha + \delta_{Tue} D^{Tue}_t + \delta_{Wed} D^{Wed}_t + \delta_{Thu} D^{Thu}_t + \delta_{Fri} D^{Fri}_t + u_t$$

Old folklore says Mondays are weak (the *Monday effect*). The question **“is there any weekday pattern at all?”** is a JOINT hypothesis — all four $\delta$s zero simultaneously — so the answer is an F-test with $q = 4$.

### 5.1 Estimate and read the coefficients

In [ ]:
y_dow = sp['SP500'] * 100          # daily return in %
m_dow = sm.OLS(y_dow, X_ok).fit()

out = pd.DataFrame({'estimate': m_dow.params, 'SE': m_dow.bse,
                    't': m_dow.tvalues, 'p': m_dow.pvalues}).round(4)
out.index = ['α (Monday mean)', 'δ_Tue', 'δ_Wed', 'δ_Thu', 'δ_Fri']
print(out)
print('\nEach δ is the average return difference of that day to Monday.')

### 5.2 Visualise — day means with confidence intervals

In [ ]:
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri']
means = [y_dow[sp['weekday'] == d].mean() for d in range(5)]
ses   = [y_dow[sp['weekday'] == d].std() / np.sqrt((sp['weekday'] == d).sum()) for d in range(5)]

fig, ax = plt.subplots(figsize=(9, 4.5))
colors = [ORANGE if m < 0 else YELLOW for m in means]
ax.bar(day_names, means, color=colors, edgecolor=GREY, lw=0.8,
       yerr=[1.96*s for s in ses], capsize=6, error_kw=dict(ecolor=GREY, lw=1.3))
ax.axhline(0, color='black', lw=1.0)
ax.set_ylabel('Average daily return (%)')
ax.set_title('S&P 500 — average return by weekday, 95% CIs', fontweight='bold', loc='left')
plt.tight_layout(); plt.show()
print('Each bar tests that day\'s MEAN against zero. That is a different hypothesis from')
print('the δ t-tests above, which test each day against MONDAY. Read both, do not mix them.')

### 5.3 The joint F-test — is there ANY pattern?

In [ ]:
print('H0: δ_Tue = δ_Wed = δ_Thu = δ_Fri = 0   (q = 4)\n')
ftest = m_dow.f_test('D_Tue = D_Wed = D_Thu = D_Fri = 0')
print(ftest)

F_crit = stats.f.ppf(0.95, 4, int(m_dow.df_resid))
print(f'\nF_crit (5%, 4, {int(m_dow.df_resid)}) = {F_crit:.2f}')
if float(np.squeeze(ftest.fvalue)) < F_crit:
    print('→ DO NOT reject H0: no significant weekday pattern.')
    print('  A test that does not reject is a result too — modern data show the anomaly has')
    print('  essentially disappeared, consistent with market efficiency.')

---
# Part 6 — The Monthly Version: A January Effect?

Same logic, twelve categories: **11 monthly dummies**, January as the reference. Each $\delta_m$ is the average return difference of month $m$ to January; the joint test has $q = 11$.

In [ ]:
sp['month'] = sp.index.month
mon_dum = pd.get_dummies(sp['month'], prefix='M', drop_first=True).astype(float)  # January dropped
X_mon = sm.add_constant(mon_dum)
m_mon = sm.OLS(y_dow, X_mon).fit()

print(f'January mean (α):  {m_mon.params["const"]:.4f}% per day')
print('\nJoint test H0: all 11 monthly δs = 0  (q = 11):')
restr = ' = '.join(mon_dum.columns) + ' = 0'
print(m_mon.f_test(restr))
print('→ Same machinery, more restrictions. In recent samples the January effect, too, is weak or absent.')

---
# Part 7 — The Chow Test: By Hand From Two R²s

The dummy-based break test IS the Chow test. Compute it from the familiar F-formula:

$$F = \frac{(R_U^2 - R_R^2)/q}{(1-R_U^2)/(n-k-1)}$$

- **Restricted**: one line for the whole sample (no dummy terms) → $R_R^2$
- **Unrestricted**: D and D·x free (each regime its own line) → $R_U^2$, $q = 2$

In [ ]:
m_R = sm.OLS(ret['AAPL'], sm.add_constant(ret['SP500'])).fit()      # restricted
m_U = sm.OLS(ret['AAPL'], X_brk).fit()                                # unrestricted (plain OLS for R²-form)

R2_R, R2_U = m_R.rsquared, m_U.rsquared
q, df_ = 2, int(m_U.df_resid)

F_hand = ((R2_U - R2_R) / q) / ((1 - R2_U) / df_)
F_crit = stats.f.ppf(0.95, q, df_)
p_hand = 1 - stats.f.cdf(F_hand, q, df_)

print(f'R²_R (one line)      = {R2_R:.4f}')
print(f'R²_U (regime lines)  = {R2_U:.4f}')
print(f'F = (({R2_U:.4f} − {R2_R:.4f})/2) / ((1 − {R2_U:.4f})/{df_}) = {F_hand:.2f}')
print(f'F_crit (5%, 2, {df_}) = {F_crit:.2f},   p = {p_hand:.4f}')
print('\nVerify with the built-in test (non-robust, to match the R²-form):')
print(m_U.f_test('D = D_x_SP500 = 0'))
print('\nCaveat: the Chow test needs a HYPOTHESISED break date. Unknown dates → recursive/CUSUM methods.')

---
## Summary Table

| Concept | Key Formula / Rule | Python |
|---------|--------------------|--------|
| Dummy | $D = 1$ if condition, else 0 | `(cond).astype(float)` |
| Intercept dummy | level shift $\beta_2$ | add `D` column |
| Slope dummy | slope shift $\beta_3$ | add `D * x` column |
| Trap rule | m categories → m − 1 dummies | `pd.get_dummies(..., drop_first=True)` |
| Reference category | coefficients = difference to base | dropped category |
| Seasonality test | joint F, $q$ = number of dummies | `model.f_test('D_Tue = … = 0')` |
| Chow test | joint F on $(D, D\cdot x)$, $q = 2$ | `model.f_test('D = D_x = 0')` |

---
*Applied Statistical Data Analysis | Prof. Dr. Kristyna Ters | FHNW School of Business | HS 2026*

*Next: Diagnostic Tests I — Heteroskedasticity & Autocorrelation.*